# Planner

> An MPC planner with communication.

In [ ]:
#| default_exp planners.mpc

In [ ]:
#| export
import torch
import torch.nn as nn
import torch.nn.functional as F

## JEPA Planner

In [ ]:
#| export
class JEPAGoalPlanner:
    def __init__(self, model, action_dim= 1, horizon=10, history_size=3,
                 pop_size=1000, topk=100, opt_steps=10, device='cpu'):
        self.model = model
        self.action_dim = action_dim
        self.horizon = horizon
        self.history_size = history_size
        self.pop_size = pop_size
        self.topk = topk
        self.opt_steps = opt_steps
        self.device = device

    def _sample_actions(self, probs):
        # probs: (B, horizon, action_dim) -> samples: (B, S, horizon, action_dim) one-hot
        B, H, A = probs.shape
        flat = probs.unsqueeze(1).expand(B, self.pop_size, H, A).reshape(-1, A)
        idx = torch.multinomial(flat, 1).view(B, self.pop_size, H)
        return F.one_hot(idx, num_classes=A).float()

    def _update_dist(self, cost, samples):
        # cost: (B, S), samples: (B, S, horizon, action_dim)
        B, S, H, A = samples.shape
        new_probs = torch.zeros(B, H, A, device=cost.device)
        for b in range(B):
            elite_idx = torch.topk(-cost[b], self.topk).indices
            elites = samples[b, elite_idx]  # (topk, H, A)
            for t in range(H):
                counts = torch.bincount(elites[:, t].argmax(dim=-1), minlength=A).float()
                new_probs[b, t] = (counts + 1e-6) / (counts.sum() + 1e-6 * A)
        return new_probs

    @torch.no_grad()
    def plan(self, info_dict):
        """
        info_dict must contain (per JEPA.get_cost / rollout requirements):
          - pixels: (B, history_size, C, H, W)   recent observation history
          - action: (B, history_size, action_dim) actions taken during that history
          - goal:   (B, C, H, W)                  goal observation
          - msg_indices (optional): (B, 1, 49)    message valid only for the first
                                                    predicted step (per model.rollout)
        Returns the first action of the best plan per batch element, plus the full plan.
        """
        device = self.device
        B = info_dict["pixels"].size(0)
        probs = torch.full((B, self.horizon, self.action_dim), 1.0 / self.action_dim, device=device)

        for _ in range(self.opt_steps):
            samples = self._sample_actions(probs)  # (B, S, horizon, action_dim)

            # full action sequence fed to rollout = known history actions + sampled future actions
            hist_action = info_dict["action"].unsqueeze(1).expand(
                B, self.pop_size, *info_dict["action"].shape[1:]
            )
            action_candidates = torch.cat([hist_action, samples], dim=2).to(device)

            # expand the rest of info_dict over the sample (S) dimension
            cand_info = {
                k: (v.unsqueeze(1).expand(B, self.pop_size, *v.shape[1:]).to(device)
                    if torch.is_tensor(v) else v)
                for k, v in info_dict.items()
            }

            cost = self.model.get_cost(cand_info, action_candidates)  # (B, S)
            probs = self._update_dist(cost, samples)

        plan = torch.argmax(probs, dim=-1)       # (B, horizon)
        first_action = plan[:, 0]
        return first_action, plan
    
    @torch.no_grad()
    def eval_plan(self, info_dict, device, plan):
        final_action_seq = torch.cat(
        [info_dict["action"], F.one_hot(plan, num_classes=self.action_dim).float()],
        dim=1,
        ).unsqueeze(1).to(device)  # (B, 1, H+horizon, action_dim)

        final_info = {
            k: (v.unsqueeze(1).to(device) if torch.is_tensor(v) else v)
            for k, v in info_dict.items()
        }
        final_cost = self.model.get_cost(final_info, final_action_seq).squeeze(1)  # (B,)

        # first_action = plan[:, 0]
        return final_cost
        